# General Overview of basic MOOSE FRAMEWORK code
 ## Language Used by Cardinal


### Goal

Build a basic understanding of how to read and write MOOSE code.


### Learning Objectives

   1. Learn MOOSE syntax and structure

   2. Learn how to execute created files with cardinal

   3. Learn Cardinal specific file structures

# Environment Setup
## Packages
We need to gather some more packages for working with MOOSE meshes
Execute the following code cell below to gather them

In [1]:
!python3 -m pip install 'pyvista[all]'

## Functions
Now we need to setup some functions we will use later, execute the following cell

In [4]:
import subprocess as sp
import tempfile as tp
import os
import pyvista as pv

def run_cardinal_file(code):
  oname="workingoutput.e"
  with tp.TemporaryDirectory(ignore_cleanup_errors=True) as work_dir:
    ifileloc = os.path.join(work_dir,"inputfile.i")
    ofileloc = os.path.join(work_dir, oname)

    with open(ifileloc,"w") as ifile:
      ifile.write(code)

    sp.run(['cardinal-opt','-i',ifileloc,'--mesh-only',ofileloc],check=True)

    ofilereader=pv.get_reader(ofileloc)
    ofile=ofilereader.read()
    ofile.plot(show_edges=True, jupyter_backend='static')


# 1. Language Overview
## Background
The MOOSE Framework was developed by INL to help provide a common base for various Finite Element Analysies.
It uses a custom language, which I will just call MOOSE or MOOSE framework, which is somewhere between python and c++

Cardinal is effectively an upgraded version of MOOSE, which also includes CFD through NEKRS and Neutron Transport through OPENMC, previously incomaptible software. It does this by creating a translation layer between MOOSE and NEKRS/OPENMC, allowing their outputs to be coupled, to give a better picture of the dynamics of the reactor and coolant.
## File Structure
### File Types
MOOSE operates on several file types, but we only work with a small subset for Cardinal
#### Input Files (.i)
**The most important filetype:**

This is where we define the structure of the MOOSE system we are using for the given sub-task.
Most simulations have multiple .i files, where each is specified for the specifc MOOSE app used.
There is often a "central" .i file which coordinate the oraganization of the file system
#### EXODOUS II Mesh Files (.e, .exo)
This filetype is what MOOSE uses to store meshes, or the data storing the shape of the simulation.
You need an external software to view this file. Examples include paraview, or pyvista, which we will use in the demo for today.
#### Model Parameter files (.par)
These files, written in MOOSE, define the variables that characterize the simulations
#### Model Setup files (.oudf, .udf)
These files are C++ code, and are used to write C++ functions for initiailizing the model (.udf), and for outputting the data (.oudf), these functions use the variables from the .par file
## Syntax
### Code Organization
MOOSE Code is broken into pre-defined major sections, and then user defined subesections. In MOOSE terminology, these are known as blocks. A block is started and ended with a set of brackets: \[\], where the first set contains the name of the block. 

For preset major blocks, examples include MESH, VARIABLES, and OUTPUT. They require no inputs, and only serve as a wrapper for user defined sub-blocks

For User defined blocks, names can contain numbers and letters, and underscores. In basic form, they require two things, the block type, and any inputs. The Type is in CamelCase. The input can take several forms, from other user blocks, to a set a values across three variables. 


An example Section block and user block:

\[MESH\]

  \[UserBlock\]

  type = SomeBlockType

  input = someotherblock

  value = somevariable

  \[\]

\[\]

### User-Stored Values 
MOOSE Code has several important-to-know formatting conventions.

#### 1. Lists and Arrays

  ##### a. Lists
  Defined by putting a set of values, with spaces in between, in a set of ''

  ex: List = '1 2 3 4 5 6'

  ##### b. Arrays
  Defined like lists, with a ; separating rows. 

  ex: Array = '1 2 3; 4 5 6'

  Stored like this:

  1 2 3

  4 5 6

  **TIP:** the array interpreter ignores line returns, so it is best practice to return after each semi-colon, and using tab to get it to match up the first value of each row, so the input matches the output

  example:

  big_array = ' 

  1 2 3 4 5 6 7;

  8 9 10 11 12 13 14;

  15 16 17 18 19 20 21
  
  '
  

#### 2. Comments
Moose does not allow multi-line comments, only single line comments.

Comments are started with a \#, and continue to right of that symbol to the end of the line

Examples:

\# Basic single line comment

someparameter = 5 \# this is also valid comment placement, as the working code is to the left of the \#

#### 3. Substitution Variables vs Variables 
  In the .i file, there are two kinds of values the user can define

  ##### a. Substitution variables
  These are better called Static Variables

  They are fixed values, defined at the top of the .i file before any blocks

  These are defined in terms of scalars, or FPparse of functions, which is written very close to plaintext math, and is from C++

  There is a very specfic formatting for fparse variables: variablename = '${fparse EXPRESSION_TO_EVALUATE}'

  These are useful for storing set values, like the sizes of certain elements, scale factors, or any fixed value that will be repeatedly referenced later

  Examples of a substitution variable section of a .i file:

  r = 5 \# Radius of circle

  c = '${fparse r^2*pi}' \# Circumference of the circle found using fparse
  ##### b. Variables
  These are better called Dynamic Variables

  These are values of a set linearity (none, constant, first, second, third, and fourth), and the family of equations it is used in. Variables are  defined under the \[Variables\] block.

  They are used to define the unknowns in any equations MOOSE will solve.

  They have a dynamic value that will change throughout the simulation, and can have an initial condition specified.

  Example of a \[Variables\] block:

  \[Varibles\]

  \[Temp\] 

  initial_condition = 80 \# Start at 80 degrees celcius

  \[\]

  \[\]
 
  **Note:** notice how temp has no set order for linearity or family, this is to allow it to be non-linear and not force MOOSE to constrain it to some curve or specific equation set. To set a linearity, we would have put order=FIRST or order=CONSTANT




# 2. Writing your first MOOSE code
We will be writing mesh generation code, as it can be executed on your local computer, and easily checked

This notebook has been setup so you can define a multi-line python string containing the MOOSE code, then enter it into a function called run_cardinal_file we defined earlier, and it will return an interface with the output of the code
## Outline

### 1. Set \[Mesh\] system block and user-block
Create a multi-line python string named firstgo,

Inside the strong, enter \[Mesh\] at the top, and \[\] on the line below

Between those two, enter \[user\] and \[\], to set the bounds of the user block
 
### 2. Set block type
We are using a block type called a GeneratedMeshGenerator, which takes a number of dimensions, the minimum and maxmimum coordinates in each direction (x,y,and z) and makes a mesh centered on the origin

Enter type = GeneratedMeshGenerator inside the user block

### 3. Set Dimensions
We want something that is two wide, therefore we want it to go from -1 to 1 on x and y

Therefore we will set xmin and ymin to -1, and xmax and ymax to 1

Do these one at a time below type = 

### 4. Set Node Counts
Meshes have a set number of defined points along their borders, called nodes. We need to set the number of sectors on sides in the x and y direction. We will use 10 as a default value, so set nx and ny to 10

### 5. Run
Now execute the code block containing run_cardinal_file, and see your results come to life



In [3]:
firstgo="""
[Mesh]
  [user]
    type = GeneratedMeshGenerator
    dim = 2
    xmin = -1
    xmax = 1
    ymin = -1
    ymax = 1
    nx=10
    ny=10
  []
[]
"""
run_cardinal_file(firstgo)

FileNotFoundError: [Errno 2] No such file or directory: 'cardinal-opt'

# 3. Working with substitution variables
Now let's do that example again, but parametrically define the max and minimum dimensions, and the node count

That way, we can tweak the values of the width and height in one location, and everything is calculated based on that automatically
## Outline

### 1. Setup code
Copy over the code from example one

### 2. Setup substitution variables
Above the Mesh block, we can start to define subsitution variables

#### a. Set the total size
We know that both the x and y width is 2, so lets make two variables, xsize and ysize

#### b. Set the min and max
We know that we are centered on the origin, so xmin and xmax are just -xsize/2 and xsize/2 respectively. Let's turn that into usable fparse code, for variables xmx and xmn

#### c. Setup node counts
We know that we had 10 nodes for a width of 2, or we have 5 nodes per unit of width. We need to create an fparse function that takes the width rounded to the nearest .2, times 5, save these to variables ncx and ncy

### Plug in Substitutions
When plugging in substitution variables, you have to format it similar to how you used fparse. Wrap it in a \$\{\}

Ex: for plugging in ncx -> nx = ${ncx}

## Run it
Execute the code block. Feel free to change xsize and ysize to see how the output changes, and experiment with modifying the formulas to see how that works


In [ ]:
secondgo="""
xsize = 2.0
ysize = 2.0

xmx = '${fparse xsize * 0.5}'
xmn = '${fparse xmx * -1}'

ymx = '${fparse ysize * 0.5}'
ymn = '${fparse ymx * -1}'

ncx = '${fparse floor(xsize/0.2) * 0.2}'
ncy = '${fparse floor(ysize/0.2) * 0.2}'
[Mesh]
  [user]
    type = GeneratedMeshGenerator
    dim = 2
    xmin = ${xmn}
    xmax = ${xmx}
    ymin = ${ymn}
    ymax = ${ymx}
    nx=${ncx}
    ny=${ncy}
  []
[]
"""
run_cardinal_file(secondgo)

# 4. Chaining User Blocks
This example will explore how to chain user blocks, which is how the user can create more complicated simulations

This will be a simple three-block example, but it will explain enough

## Outline

### 1. Setup a basic Mesh block
Just as before make the string (let's call it thirdgo) and the mesh block pair

### 2. Define the 3 user blocks
Make three user blocks, called ShapeOne, ShapeTwo, and ShapeFinal

### 3. Fill in the details for the ShapeOne and ShapeTwo
They should be both GeneratedMeshGenerators in 2d, feel free to make them whatever size you want, just make them unique

### 4. Define the ShapeFinal block

This will be a new type of generator, TransformGenerator, which allows us to translate or rotate a given mesh, either relatively or to an absolute position or rotation. We will doing a relative rotation of 30 degrees about the z axis

#### a. Define the transform type
Put in the type, and the input as ShapeTwo
Then put the value transform = ROTATE

#### b. Input the rotation
Then we input the rotation in a list called vector_value, where the values are 'phi theta psi', or rotations about the x,y,z axes, we want 0 0 30 in this case

### 5. Select generator
 MOOSE maps the chains of user blocks in a tree-branch style. If we were to run the code as it is right now, it will throw an error, as it has multiple paths it could follow, ending at ShapeOne and at ShapeFinal. Without a set end point it doesn't know which branch to follow. We can tell it by putting a final_generator = above the first user block in \[Mesh\], since we want to end at ShapeFinal, we would put final_generator = ShapeFinal 

## Run it
Feel free to tweak the rotation vector to see what happens, also feel free to switch which shape we are pulling from

In [ ]:
thirdgo="""
[Mesh]
  final_generator = ShapeFinal
  [ShapeOne]
    type = GeneratedMeshGenerator
    dim = 2
    xmin = -3
    xmax = 1
    ymin = -1
    ymax = 1
    nx=10
    ny=10
  []
  [ShapeTwo]
    type = GeneratedMeshGenerator
    dim = 2
    xmin = 0.5
    xmax = 3
    ymin = -1
    ymax = 1
    nx=10
    ny=10
  []
  [ShapeFinal]
    type = TransformGenerator
    input = ShapeTwo
    transform = ROTATE
    vector_value = '0 0 30'
  []


[]
"""
run_cardinal_file(thirdgo)

# Summary Checklist
| You Should be Able to | Section |
|-----------------------|---------|
| Understand MOOSE file structure | 1 |
| Understand MOOSE file organization | 1 |
| Be able to write user MOOSE blocks | 2 |
| Understand how to use substitution variables | 3 |
| Understand how to chain MOOSE blocks together | 4 |
| Understand how to manually set branch selection | 4 |

